# Notebook 4: Tools

**What you'll learn:**
- What tools are and why agents need them
- How `@tool` transforms a function into an AgentTool
- What ToolSpec, ToolUse, and ToolResult are
- How ToolRegistry stores and manages tools
- How to use ToolContext to access agent state from inside a tool
- Parallel vs sequential tool execution

**Prerequisite:** Complete [NB3_Models.ipynb](./NB3_Models.ipynb)

**Companion reading:** `04-tools.md`

---
## What are Tools?

Without tools, an agent can only **talk**. With tools, it can **act**.

Think of tools as **buttons on a wall**. Each button:
- Has a label (name) and description
- Requires specific inputs (parameters)
- Produces a result when pressed

The model reads the button labels and descriptions, decides which to press, fills in the inputs, and reads the result.

### The Tool Lifecycle

```
1. DEFINE      You write a Python function with @tool
       |
2. REGISTER    You pass it to Agent(tools=[...])
       |
3. SPECIFY     SDK converts it to a JSON schema (ToolSpec) for the model
       |
4. MODEL SEES  Model reads ToolSpec to know what tools exist
       |
5. MODEL CALLS Model sends ToolUse ("I want to call this tool with these args")
       |
6. EXECUTE     SDK calls your function with the model's arguments
       |
7. RESULT      ToolResult is sent back to the model
```

In [ ]:
# ============================================================
# STEP 1: Define a tool and inspect it
# ============================================================

from strands import tool

# The @tool decorator transforms this plain Python function
# into an AgentTool object. It reads:
#   - Function name -> tool_name
#   - Docstring -> description
#   - Type hints -> parameter schema

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city to check weather for.
    """
    # In real code, this would call a weather API.
    # For learning, we return a fake response.
    return f"72F and sunny in {city}"

# After @tool, get_weather is NOT a plain function anymore.
# It's a DecoratorTool object with special attributes.
print(f"Type:      {type(get_weather).__name__}")  # DecoratorTool
print(f"Tool name: {get_weather.tool_name}")        # "get_weather"
print(f"Tool type: {get_weather.tool_type}")        # "python"

In [ ]:
# ============================================================
# STEP 2: Look at the ToolSpec (what the model sees)
# ============================================================

import json

# The tool_spec is a JSON schema that tells the model:
#   - What the tool is called
#   - What it does (from the docstring)
#   - What parameters it accepts (from type hints)
#   - Which parameters are required
#
# The model uses this schema to decide whether to call the tool
# and what arguments to pass.

spec = get_weather.tool_spec
print("=== ToolSpec (what the model sees) ===")
print(json.dumps(spec, indent=2))

### How @tool Works (Function Signature -> JSON Schema)

The `@tool` decorator reads your function and automatically generates the ToolSpec:

| Python | ToolSpec field |
|--------|---------------|
| Function name: `get_weather` | `"name": "get_weather"` |
| First line of docstring | `"description": "Get the current weather..."` |
| Parameter `city: str` | `"properties": {"city": {"type": "string"}}` |
| Args section in docstring | `"description": "The name of the city..."` |
| No default value for `city` | `"required": ["city"]` |

**Source:** `src/strands/tools/decorator.py`

In [ ]:
# ============================================================
# STEP 3: More detailed tool -- see how the spec changes
# ============================================================

# A more detailed tool with multiple parameters and types.
# Notice how the docstring Args section maps to parameter descriptions.

@tool
def search_products(
    query: str,
    max_results: int = 5,
    category: str = "all",
) -> str:
    """Search for products in the catalog.
    
    Use this tool when the user wants to find products by keyword.
    
    Args:
        query: The search term to look for.
        max_results: Maximum number of results to return.
        category: Product category to filter by.
    """
    return f"Found {max_results} products for '{query}' in '{category}'"

print("=== search_products ToolSpec ===")
print(json.dumps(search_products.tool_spec, indent=2))

# Notice:
# - 'query' is required (no default value)
# - 'max_results' and 'category' are NOT required (they have defaults)
# - The docstring provides descriptions for each parameter

---
## ToolRegistry -- Where Tools Live

When you create `Agent(tools=[get_weather, search_products])`, the agent stores the tools in a **ToolRegistry**.

The ToolRegistry is a dictionary: `{tool_name: AgentTool}`. It provides:
- `registry` -- the dict of all tools
- `get_all_tool_specs()` -- returns all ToolSpecs for the model

**Source:** `src/strands/tools/registry.py`

In [ ]:
# ============================================================
# STEP 4: Inspect the ToolRegistry
# ============================================================

from strands import Agent

# Create agent with both tools
agent = Agent(
    tools=[get_weather, search_products],
    callback_handler=None,  # Quiet mode
)

# The tool_registry stores all registered tools.
registry = agent.tool_registry

# List all registered tool names
print("=== Registered Tools ===")
for name in registry.registry.keys():
    print(f"  - {name}")

print()

# get_all_tool_specs() returns the specs sent to the model.
# This is what the model reads to know what tools are available.
specs = registry.get_all_tool_specs()
print(f"=== ToolSpecs for model ({len(specs)} tools) ===")
for spec in specs:
    print(f"  - {spec['name']}: {spec.get('description', '')[:60]}")

In [ ]:
# ============================================================
# STEP 5: Call the agent and see toolUse + toolResult
# ============================================================

# Ask a question that should trigger the get_weather tool.
result = agent("What's the weather in Seattle?")

print(f"Result: {result}")
print(f"\n=== Message History ===")
print(f"Total messages: {len(agent.messages)}")
print()

# Walk through each message to see the toolUse and toolResult
for i, msg in enumerate(agent.messages):
    role = msg['role']
    print(f"[{i}] {role}")
    
    for block in msg['content']:
        if 'text' in block:
            print(f"     text: \"{block['text'][:80]}\"")
        elif 'toolUse' in block:
            # toolUse = the model's request to call a tool
            # Contains: toolUseId (unique ID), name, input (arguments)
            tu = block['toolUse']
            print(f"     toolUse: {tu['name']}({json.dumps(tu['input'])})")
            print(f"              toolUseId: {tu['toolUseId']}")
        elif 'toolResult' in block:
            # toolResult = the result of executing the tool
            # Contains: toolUseId (matches the request), status, content
            tr = block['toolResult']
            text = ""
            for item in tr.get('content', []):
                if 'text' in item:
                    text = item['text']
            print(f"     toolResult: \"{text}\"")
            print(f"                 status: {tr.get('status')}")
    print()

---
## ToolContext -- Accessing Agent State From Inside a Tool

Sometimes your tool needs to access the agent's state (e.g., to store data for later). The `ToolContext` provides this access.

To use it, add `tool_context` as a parameter to your tool function. The SDK automatically fills it in.

**ToolContext provides:**
- `tool_context.agent` -- the Agent instance
- `tool_context.agent.state` -- dictionary for storing custom data
- `tool_context.agent.messages` -- conversation history
- `tool_context.interrupt()` -- trigger human-in-the-loop pause

**Source:** `src/strands/types/tools.py`

In [ ]:
# ============================================================
# STEP 6: Tool with ToolContext
# ============================================================

from strands.types.tools import ToolContext

# When you add 'tool_context: ToolContext' as a parameter,
# the SDK automatically passes it in. The model does NOT
# see this parameter -- it's hidden from the ToolSpec.

@tool
def save_note(title: str, content: str, tool_context: ToolContext) -> str:
    """Save a note for the user.
    
    Args:
        title: The title of the note.
        content: The content of the note.
    """
    # Access the agent's state dictionary.
    # This persists across tool calls within the same agent.
    notes = tool_context.agent.state.setdefault("notes", [])
    notes.append({"title": title, "content": content})
    
    return f"Saved note '{title}' (total notes: {len(notes)})"

# Check the ToolSpec -- notice tool_context is NOT in the schema.
# The model doesn't know about it; the SDK handles it internally.
print("=== save_note ToolSpec ===")
print(json.dumps(save_note.tool_spec, indent=2))

In [ ]:
# ============================================================
# STEP 7: Use the context-aware tool
# ============================================================

# Create a new agent with the save_note tool
agent_notes = Agent(
    tools=[save_note],
    callback_handler=None,
)

# Call the agent -- it should use save_note
result = agent_notes("Save a note titled 'Meeting' with content 'Discuss Q1 roadmap'")
print(f"Result: {result}")
print()

# Check the agent's state -- the note should be there.
# The tool used tool_context.agent.state to store it.
print("=== Agent State ===")
print(json.dumps(agent_notes.state, indent=2))

---
## Multiple Tools -- The Model Picks

When you give an agent multiple tools, the model decides which to call based on the user's request. It reads all ToolSpecs and picks the most relevant one(s).

In [ ]:
# ============================================================
# STEP 8: Multiple tools -- model picks the right one(s)
# ============================================================

@tool
def get_time(timezone: str) -> str:
    """Get the current time in a timezone.
    
    Args:
        timezone: The timezone name, e.g. 'US/Pacific'.
    """
    from datetime import datetime
    # Simplified -- returns local time regardless of timezone
    return f"The current time in {timezone} is {datetime.now().strftime('%I:%M %p')}"

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers.
    
    Args:
        a: First number.
        b: Second number.
    """
    return str(a * b)

# Agent with 3 tools
agent_multi = Agent(
    tools=[get_weather, get_time, multiply],
    callback_handler=None,
)

# Ask a question that needs 2 tools (weather + time)
result = agent_multi("What's the weather and current time in Seattle?")
print(f"Result: {result}")
print()

# Check which tools were called by inspecting messages
print("=== Tools Called ===")
for msg in agent_multi.messages:
    for block in msg['content']:
        if 'toolUse' in block:
            tu = block['toolUse']
            print(f"  {tu['name']}({json.dumps(tu['input'])})")

---
## Parallel vs Sequential Tool Execution

When the model requests multiple tools in one cycle, the SDK can execute them **in parallel** (at the same time) using a thread pool. This is faster than running them one after another.

By default, parallel execution is enabled. The SDK uses `ThreadPoolExecutor` to run tools concurrently.

**Source:** `src/strands/tools/executors/`

In [ ]:
# ============================================================
# STEP 9: Parallel execution demo
# ============================================================

import time

# These tools simulate slow operations (1 second each)
@tool
def slow_task_a(input_text: str) -> str:
    """Perform slow task A.
    
    Args:
        input_text: Input for the task.
    """
    time.sleep(1)  # Simulate 1 second of work
    return f"Task A done with: {input_text}"

@tool
def slow_task_b(input_text: str) -> str:
    """Perform slow task B.
    
    Args:
        input_text: Input for the task.
    """
    time.sleep(1)  # Simulate 1 second of work
    return f"Task B done with: {input_text}"

agent_parallel = Agent(
    tools=[slow_task_a, slow_task_b],
    callback_handler=None,
)

# Time the call. If parallel, both 1-second tasks should finish in ~1 second.
# If sequential, it would take ~2 seconds.
start = time.time()
result = agent_parallel("Run both slow_task_a and slow_task_b with input 'hello'")
elapsed = time.time() - start

print(f"Result: {result}")
print(f"\nTime elapsed: {elapsed:.1f} seconds")
print(f"(If parallel: ~1s + model time. If sequential: ~2s + model time)")

---
## Summary

What you learned in this notebook:

- **Tools** let agents act, not just talk -- they're like buttons the model can press
- **@tool decorator** transforms a function into an AgentTool by reading name, docstring, and type hints
- **ToolSpec** is the JSON schema the model reads to know what tools exist
- **ToolUse** is the model's request to call a tool (name + arguments)
- **ToolResult** is the output sent back to the model
- **ToolRegistry** stores tools by name: `agent.tool_registry.registry`
- **ToolContext** gives tools access to the agent (state, messages, interrupts)
- **Parallel execution** runs multiple tools at the same time for speed

**Key source files:**
| File | What it does |
|------|--------------|
| `src/strands/tools/decorator.py` | @tool decorator, DecoratorTool |
| `src/strands/tools/registry.py` | ToolRegistry |
| `src/strands/types/tools.py` | ToolSpec, ToolUse, ToolResult, ToolContext |
| `src/strands/tools/executors/` | Thread pool executor |

**Next:** [NB5_Hooks.ipynb](./NB5_Hooks.ipynb) -- The hook system: lifecycle events, logging, guards, and retries